<a href="https://colab.research.google.com/github/dannyboy6661984/Health-Logger/blob/main/HealthDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Health Data DB
## scripts to build and populate my sql database

### Install dependencies


In [ ]:
!pip install garminconnect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.1/66.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


###Import libraries


In [ ]:
import sqlite3
import pandas as pd
import getpass
from google.colab import files
from datetime import datetime, timedelta
from garminconnect import Garmin

### Define Variables

In [ ]:
# Initialize database and tables with correct schema
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON;')

# 1. Create master dates table
cursor.execute('''
CREATE TABLE IF NOT EXISTS master_dates (
    date_id TEXT PRIMARY KEY
);
''')

# 2. Create daily steps table (including metadata columns from the start)
cursor.execute('''
CREATE TABLE IF NOT EXISTS daily_steps (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    date_id TEXT UNIQUE,
    step_count INTEGER NOT NULL,
    last_updated TEXT,
    status TEXT DEFAULT 'PENDING',
    FOREIGN KEY (date_id) REFERENCES master_dates(date_id)
);
''')

conn.commit()
print("Database and tables initialized successfully with upgraded schema!")

# File and Export Configuration
local_db_filename = 'health.db'

# Query Configuration
query = '''
SELECT d.date_id, s.step_count, s.last_updated, s.status
FROM master_dates d
LEFT JOIN daily_steps s ON d.date_id = s.date_id
ORDER BY d.date_id DESC;
'''

Database and tables initialized successfully with upgraded schema!


### Define Functions


In [ ]:
def full_sync_overwrite_all(client, total_days_history=30):
    """
    Overwrites and populates all database records for the last `total_days_history` days.
    Useful for initial setup or total recovery.
    """
    today = datetime.now()
    print(f"Starting full overwrite sync for the last {total_days_history} days...")

    for i in range(total_days_history):
        target_date = today - timedelta(days=i)
        date_str = target_date.strftime('%Y-%m-%d')

        # Today remains PENDING, historical dates are written as FINALIZED
        status = 'PENDING' if i < 3 else 'FINALIZED'

        try:
            stats = client.get_user_summary(date_str)
            steps = stats.get('totalSteps', 0)

            cursor.execute('INSERT OR IGNORE INTO master_dates (date_id) VALUES (?);', (date_str,))
            cursor.execute('''
            INSERT OR REPLACE INTO daily_steps (date_id, step_count, last_updated, status)
            VALUES (?, ?, ?, ?);
            ''', (date_str, steps, datetime.now().strftime('%Y-%m-%d %H:%M:%S'), status))

            print(f"Overwrote {date_str}: {steps} steps ({status})")
        except Exception as e:
            print(f"Failed to fetch {date_str}: {e}")

    conn.commit()
    print("Full sync complete!")

In [ ]:
def incremental_sync_unfinalized(client, check_days=30):
    """
    Checks the last `check_days`. Updates any day that is not yet 'FINALIZED'.
    Seals a day's status to 'FINALIZED' if current time is > 72 hours after that day's midnight.
    """
    now = datetime.now()
    print("Starting incremental sync for unfinalized dates...")

    # Fetch existing statuses
    cursor.execute("SELECT date_id, status FROM daily_steps;")
    existing_status = {row[0]: row[1] for row in cursor.fetchall()}

    for i in range(check_days):
        target_date = now - timedelta(days=i)
        date_str = target_date.strftime('%Y-%m-%d')

        current_status = existing_status.get(date_str, None)

        # We only sync if the status is PENDING or doesn't exist yet
        if current_status is None or current_status == 'PENDING':
            target_end_of_day = datetime.combine(target_date.date(), datetime.max.time())

            if now > target_end_of_day + timedelta(hours=72):
                next_status = 'FINALIZED'
            else:
                next_status = 'PENDING'

            try:
                stats = client.get_user_summary(date_str)
                steps = stats.get('totalSteps', 0)

                cursor.execute('INSERT OR IGNORE INTO master_dates (date_id) VALUES (?);', (date_str,))
                cursor.execute('''
                INSERT OR REPLACE INTO daily_steps (date_id, step_count, last_updated, status)
                VALUES (?, ?, ?, ?);
                ''', (date_str, steps, now.strftime('%Y-%m-%d %H:%M:%S'), next_status))

                print(f"Synced & Updated {date_str}: {steps} steps (New Status: {next_status})")
            except Exception as e:
                print(f"Failed to sync {date_str}: {e}")

    conn.commit()
    print("Incremental sync complete!")

In [ ]:
# Safe credentials collector and client authentication
username = input("Enter your Garmin Connect Email: ")
password = getpass.getpass("Enter your Garmin Connect Password: ")

try:
    client = Garmin(username, password)
    client.login()
    print("Successfully authenticated with Garmin Connect!")
except Exception as e:
    print(f"Error connecting to Garmin: {e}")

Enter your Garmin Connect Email: dan_watkin@hotmail.com
Enter your Garmin Connect Password: ··········


Successfully authenticated with Garmin Connect!


### Interfacing with Garmin Connect (Unofficial API Wrapper)

First, we will install the `garminconnect` library. This library acts as a wrapper around the Garmin Connect web portal endpoints.

Now, we can authenticate and fetch the steps. For your privacy, we will use `getpass` to let you securely enter your password when running the cell.

In [ ]:
# Run the full backfill sync (e.g., last 14 days)
full_sync_overwrite_all(client, total_days_history=14)

Starting full overwrite sync for the last 14 days...
Overwrote 2026-10-23: 3465 steps (PENDING)
Overwrote 2026-10-22: 14145 steps (PENDING)
Overwrote 2026-10-21: 14232 steps (PENDING)
Overwrote 2026-10-20: 13069 steps (FINALIZED)
Overwrote 2026-10-19: 9225 steps (FINALIZED)
Overwrote 2026-10-18: 12317 steps (FINALIZED)
Overwrote 2026-10-17: 17742 steps (FINALIZED)
Overwrote 2026-10-16: 12178 steps (FINALIZED)
Overwrote 2026-10-15: 19697 steps (FINALIZED)
Overwrote 2026-10-14: 8670 steps (FINALIZED)
Overwrote 2026-10-13: 12027 steps (FINALIZED)
Overwrote 2026-10-12: 14292 steps (FINALIZED)
Overwrote 2026-10-11: 16784 steps (FINALIZED)
Overwrote 2026-10-10: 20147 steps (FINALIZED)
Full sync complete!


In [ ]:
# View the results in the database using the defined query
df_results = pd.read_sql_query(query, conn)
display(df_results)

,date_id,step_count,last_updated,status
0,2026-10-23,3465,2026-10-23 09:46:07,PENDING
1,2026-10-22,14145,2026-10-23 09:46:08,FINALIZED
2,2026-10-21,14232,2026-10-23 09:46:08,FINALIZED
3,2026-10-20,13069,2026-10-23 09:46:08,FINALIZED
4,2026-10-19,9225,2026-10-23 09:46:08,FINALIZED
5,2026-10-18,12317,2026-10-23 09:46:08,FINALIZED
6,2026-10-17,17742,2026-10-23 09:46:08,FINALIZED
7,2026-10-16,12178,2026-10-23 09:46:08,FINALIZED
8,2026-10-15,19697,2026-10-23 09:46:08,FINALIZED
9,2026-10-14,8670,2026-10-23 09:46:08,FINALIZED


In [ ]:
# Export the database locally and trigger a download using defined local_db_filename
file_conn = sqlite3.connect(local_db_filename)
with file_conn:
    conn.backup(file_conn)
file_conn.close()
print(f"Database successfully exported to {local_db_filename}!")

files.download(local_db_filename)

Table: master_dates
CREATE TABLE master_dates (
    date_id TEXT PRIMARY KEY
)
--------------------------------------------------
Table: daily_steps
CREATE TABLE daily_steps (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    date_id TEXT UNIQUE,
    step_count INTEGER NOT NULL,
    FOREIGN KEY (date_id) REFERENCES master_dates(date_id)
)
--------------------------------------------------


### Upgrading Table Schema to Track Sync Status

We will modify the database by adding two new columns to the `daily_steps` table:
- `last_updated`: A timestamp of when the step count was last synced.
- `status`: Mark it as `'PENDING'` (for today/yesterday where steps can still grow) or `'FINALIZED'` (for older dates).

In [ ]:
# Since SQLite doesn't easily support dropping/re-adding foreign key columns without rebuilding,
# we will simply alter our existing daily_steps table to add the new metadata columns.

try:
    cursor.execute("ALTER TABLE daily_steps ADD COLUMN last_updated TEXT;")
    cursor.execute("ALTER TABLE daily_steps ADD COLUMN status TEXT DEFAULT 'PENDING';")
    conn.commit()
    print("Schema upgraded successfully!")
except sqlite3.OperationalError as e:
    # In case columns already exist
    print(f"Database already has upgraded columns: {e}")

Schema upgraded successfully!


### Smart Sync Function

Now, let's create a reusable function that syncs a configurable range of days (e.g., the last 3 days to catch any late-syncing Garmin data), updates the step counts, and automatically marks days older than 24 hours as `'FINALIZED'`.

In [ ]:
# Verify the results in our database
query = '''
SELECT d.date_id, s.step_count, s.last_updated, s.status
FROM master_dates d
LEFT JOIN daily_steps s ON d.date_id = s.date_id
ORDER BY d.date_id DESC;
'''
display(pd.read_sql_query(query, conn))

,date_id,step_count,last_updated,status
0,2026-10-21,5969,2026-10-21 14:36:58,PENDING
1,2026-10-20,13069,2026-10-21 14:36:58,FINALIZED
2,2026-10-19,9225,2026-10-21 14:36:58,FINALIZED
3,2026-10-18,12000,None,PENDING
4,2026-10-17,7500,None,PENDING


### 1. Full Overwrite Sync Function
This function is perfect for initial setup. It loops back for a specified number of days (e.g., 30 days), fetches all step metrics from Garmin, and inserts them as `'FINALIZED'` (except for today, which stays `'PENDING'`).

In [ ]:
def sync_garmin_steps(client, days_lookback=34):
    """
    Syncs step data for the last 'days_lookback' days.
    Marks today's record as 'PENDING' and older records as 'FINALIZED'.
    """
    today = datetime.now()

    for i in range(days_lookback):
        target_date = today - timedelta(days=i)
        date_str = target_date.strftime('%Y-%m-%d')

        status = 'PENDING' if i == 0 else 'FINALIZED'

        try:
            stats = client.get_user_summary(date_str)
            steps = stats.get('totalSteps', 0)

            cursor.execute('INSERT OR IGNORE INTO master_dates (date_id) VALUES (?);', (date_str,))
            cursor.execute('''
            INSERT OR REPLACE INTO daily_steps (date_id, step_count, last_updated, status)
            VALUES (?, ?, ?, ?);
            ''', (date_str, steps, datetime.now().strftime('%Y-%m-%d %H:%M:%S'), status))

            print(f"Synced {date_str}: {steps} steps ({status})")
        except Exception as e:
            print(f"Failed to sync {date_str}: {e}")

    conn.commit()

### 2. Incremental Smart Sync Function
This function checks for any records still marked as `'PENDING'` (or completely missing in your master list). If the date is older than 72 hours (3 days ago from now), it automatically locks it to `'FINALIZED'`.

In [ ]:
# Execute a full sync for the last 30 days of Garmin data
full_sync_overwrite_all(client, total_days_history=34)

# Query and display the results to confirm your database has been completely backfilled!
query = '''
SELECT d.date_id, s.step_count, s.last_updated, s.status
FROM master_dates d
LEFT JOIN daily_steps s ON d.date_id = s.date_id
ORDER BY d.date_id DESC;
'''
df_results = pd.read_sql_query(query, conn)
display(df_results)

Starting full overwrite sync for the last 34 days...
Overwrote 2026-10-23: 3465 steps (PENDING)
Overwrote 2026-10-22: 14145 steps (FINALIZED)
Overwrote 2026-10-21: 14232 steps (FINALIZED)
Overwrote 2026-10-20: 13069 steps (FINALIZED)
Overwrote 2026-10-19: 9225 steps (FINALIZED)
Overwrote 2026-10-18: 12317 steps (FINALIZED)
Overwrote 2026-10-17: 17742 steps (FINALIZED)
Overwrote 2026-10-16: 12178 steps (FINALIZED)
Overwrote 2026-10-15: 19697 steps (FINALIZED)
Overwrote 2026-10-14: 8670 steps (FINALIZED)
Overwrote 2026-10-13: 12027 steps (FINALIZED)
Overwrote 2026-10-12: 14292 steps (FINALIZED)
Overwrote 2026-10-11: 16784 steps (FINALIZED)
Overwrote 2026-10-10: 20147 steps (FINALIZED)
Overwrote 2026-10-09: 12178 steps (FINALIZED)
Overwrote 2026-10-08: 14851 steps (FINALIZED)
Overwrote 2026-10-07: 12379 steps (FINALIZED)
Overwrote 2026-10-06: 16621 steps (FINALIZED)
Overwrote 2026-10-05: 12065 steps (FINALIZED)
Overwrote 2026-10-04: 13414 steps (FINALIZED)
Overwrote 2026-10-03: 24537 step

,date_id,step_count,last_updated,status
0,2026-10-23,3465,2026-10-23 09:47:03,PENDING
1,2026-10-22,14145,2026-10-23 09:47:03,FINALIZED
2,2026-10-21,14232,2026-10-23 09:47:03,FINALIZED
3,2026-10-20,13069,2026-10-23 09:47:03,FINALIZED
4,2026-10-19,9225,2026-10-23 09:47:03,FINALIZED
5,2026-10-18,12317,2026-10-23 09:47:03,FINALIZED
6,2026-10-17,17742,2026-10-23 09:47:03,FINALIZED
7,2026-10-16,12178,2026-10-23 09:47:03,FINALIZED
8,2026-10-15,19697,2026-10-23 09:47:03,FINALIZED
9,2026-10-14,8670,2026-10-23 09:47:03,FINALIZED


In [ ]:
!git --version

git version 2.43.0


In [ ]:
!git config --global user.name "DPW"

In [ ]:
!git config --list

user.name=DPW
user.email=your.email@example.com


In [ ]:

!git config --global user.email "your.email@example.com"

!git config --list --show-origin

file:/root/.gitconfig	user.name=DPW
file:/root/.gitconfig	user.email=your.email@example.com


In [ ]:
!git config --global user.name "Your Name"
!git config --global user.email "your.email@example.com"

!git config --list --show-origin

file:/root/.gitconfig	user.name=Your Name
file:/root/.gitconfig	user.email=your.email@example.com


In [ ]:
!git clone git@github.com:dannyboy6661984/Health-Logger

Cloning into 'Health-Logger'...
Host key verification failed.
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.


In [ ]:
!ls

!cd healthdb

!ls

git  healthdb  sample_data
git  healthdb  sample_data


In [ ]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


In [ ]:
!cd healthdb/
!git init

Reinitialized existing Git repository in /content/.git/


In [ ]:
!ls -a
!cd healthdb/
!ls -a

.  ..  .config	.git  git  healthdb  sample_data
.  ..  .config	.git  git  healthdb  sample_data
